In [13]:
import argparse
import pyscf
import ffsim
import numpy as np
import scipy

import sys

sys.path.append("../")

from optimize import commutator_cost, variance_cost

In [5]:
molpath = "../hamiltonians/H4_linear_d2.0670.chk"

mol = pyscf.lib.chkfile.load_mol(molpath)
mf = pyscf.scf.RHF(mol)
mf.update_from_chk(molpath)
moldata = ffsim.MolecularData.from_scf(mf)

In [6]:
h_linop = ffsim.linear_operator(moldata.hamiltonian,
                                    norb=moldata.norb,
                                    nelec=moldata.nelec)

In [11]:
h_linop

<36x36 _CustomLinearOperator with dtype=complex128>

In [14]:
x = np.random.randn(moldata.norb * (moldata.norb - 1) // 2)

iu = np.triu_indices(moldata.norb, k=1)
rotation_generator = np.zeros((moldata.norb, moldata.norb))
rotation_generator[iu] = x
rotation_generator -= rotation_generator.T
U = scipy.linalg.expm(rotation_generator)

In [18]:
rotated_h = moldata.hamiltonian.rotated(U)

In [19]:
rotated_h_linop = ffsim.linear_operator(rotated_h,
                                    norb=moldata.norb,
                                    nelec=moldata.nelec)

In [20]:
e_0, v_0 = scipy.sparse.linalg.eigsh(h_linop, which="SA", k=1)
print(e_0)

[-1.89169532]


In [21]:
e_rotated, v_rotated = scipy.sparse.linalg.eigsh(rotated_h_linop, which="SA", k=1)
print(e_rotated)

[-1.89169532]


In [27]:
abs(v_rotated.T.conj() @ ffsim.apply_orbital_rotation(v_0, U, moldata.norb, moldata.nelec))

array([1.])

We need orbital seniority projectors or something like that. Then a subspace Hamiltonian will just be sandwiched between the projectors

In [30]:
def generalized_seniority_and_constant(i, a, b, c, d):
    return ffsim.FermionOperator(
                {
                    (ffsim.cre_a(i), ffsim.des_a(i)): a,
                    (ffsim.cre_b(i), ffsim.des_b(i)): b,
                    (ffsim.cre_a(i), ffsim.des_a(i), ffsim.cre_b(i), ffsim.des_b(i)): c,
                    (): d
                }
            )

In [31]:
generalized_seniority_and_constant(1, -1., -1., 1., 1.)

FermionOperator({
    (cre_b(1), des_b(1)): -1,
    (): 1,
    (cre_a(1), des_a(1)): -1,
    (cre_a(1), des_a(1), cre_b(1), des_b(1)): 1
})

By fixing $(a, b, c)$, we are partitioning the Hilbert space into subspaces based on populations of $n_{i \alpha}, n_{i \beta}$. There are 15 possible partitions, and all are attainable with some nonzero choices of $(a, b, c)$, except for the trivial partition which requires $a = b = c = 0$. Do I have to track 14 possible cases then?..

In [46]:
def distinct_local_sectors(a, b, c, atol=1e-3):
    sector_eigenvalues = [0, a, b, a + b + c]

    # https://stackoverflow.com/a/38924644/
    partitions = [] # Found partitions
    for i, e in enumerate(sector_eigenvalues): # Loop over each element
        found = False # Note it is not yet part of a known partition
        for p in partitions:
            if np.isclose(e, sector_eigenvalues[p[0]], atol=atol): # Found a partition for it!
                p.append(i)
                found = True
                break
        if not found: # Make a new partition for it.
            partitions.append([i])
    return partitions

In [52]:
def generalized_seniority_projectors(orbital_index, a, b, c):
    sectors = distinct_local_sectors(a, b, c)
    projectors = []
    for sector in sectors:
        operator = ffsim.FermionOperator({(): 0})
        for j in sector:
            if j == 0:
                operator += generalized_seniority_and_constant(orbital_index, -1, -1, 1, 1)
            elif j == 1:
                operator += generalized_seniority_and_constant(orbital_index, 1, 0, -1, 0)
            elif j == 2:
                operator += generalized_seniority_and_constant(orbital_index, 0, 1, -1, 0)
            elif j == 3:
                operator += generalized_seniority_and_constant(orbital_index, 0, 0, 1, 0)
            else:
                raise ValueError()
        projectors.append(operator)
    return projectors

In [58]:
generalized_seniority_projectors(0, 1, 1.1, -1.5)

[FermionOperator({
    (cre_a(0), des_a(0)): -1,
    (): 1,
    (cre_a(0), des_a(0), cre_b(0), des_b(0)): 1,
    (cre_b(0), des_b(0)): -1
}),
 FermionOperator({
    (cre_a(0), des_a(0)): 1,
    (): 0,
    (cre_b(0), des_b(0)): 0,
    (cre_a(0), des_a(0), cre_b(0), des_b(0)): -1
}),
 FermionOperator({
    (): 0,
    (cre_a(0), des_a(0)): 0,
    (cre_b(0), des_b(0)): 1,
    (cre_a(0), des_a(0), cre_b(0), des_b(0)): -1
}),
 FermionOperator({
    (cre_a(0), des_a(0), cre_b(0), des_b(0)): 1,
    (): 0,
    (cre_a(0), des_a(0)): 0,
    (cre_b(0), des_b(0)): 0
})]